# ResonantBERT v2 Evaluation — Local Machine
## 4 Baselines + ResonantBERT v2 on the same held-out test set

**Baselines:**
1. Rule-Based (keyword heuristics — no learning)
2. TF-IDF + Ridge (classical ML, trained on the train split)
3. Sentence-BERT (pretrained embeddings, no task-specific training)
4. LLM Zero-Shot (local instruct LLM, prompt-based scoring) — or a clearly-labelled keyword heuristic if `USE_LLM = False`

**Ours:**
5. ResonantBERT v2 + RankingTower v2 (checkpoints in `./model`)

**Before running** (from the repo root):
```
git lfs pull                      # fetches bbc_articles/ranking.json + articles
pip install torch --index-url https://download.pytorch.org/whl/cu128
pip install transformers peft sentence-transformers scikit-learn scipy pyyaml pandas matplotlib seaborn
```

**Output:** paper-ready tables (LaTeX), figures (PNG), significance tests → `./eval_results`.

In [ ]:
# ============================================================
# SETUP
# ============================================================
import torch, types

# transformers 5.x imports a symbol newer torch removed (same patch as training)
if not hasattr(torch, "_utils"):
    torch._utils = types.ModuleType("torch._utils")
if not hasattr(torch._utils, "_get_device_index"):
    def _get_device_index(device, optional=False, allow_cpu=False):
        if device is None:
            if optional: return None
            raise ValueError("device is None")
        if isinstance(device, str): device = torch.device(device)
        if isinstance(device, torch.device):
            if device.type == "cpu":
                if allow_cpu: return -1
                raise RuntimeError("CPU not allowed")
            return device.index if device.index is not None else 0
        return device
    torch._utils._get_device_index = _get_device_index

import os, re, json, sys, glob, pickle, random, warnings
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import kendalltau, ttest_rel, wilcoxon
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# Paths are relative to the repo root (where this notebook lives)
REPO_DIR       = os.path.abspath(".")
CHECKPOINT_DIR = os.path.join(REPO_DIR, "model")         # BERTv2-*.pt, RANKERv2-*.pt
BBC_DIR        = os.path.join(REPO_DIR, "bbc_articles")  # articles/*.yaml + ranking.json (git lfs pull)
OUTPUT_DIR     = os.path.join(REPO_DIR, "eval_results")
CACHE_DIR      = os.path.join(OUTPUT_DIR, "cache")
os.makedirs(CACHE_DIR, exist_ok=True)

# Must match the training notebook exactly (determines the test split)
SEED           = 42
TOKENIZER_NAME = "answerdotai/ModernBERT-base"
MAX_SEQ_LEN    = 320
EMBEDDING_DIM  = 128
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
VAL_FRAC       = 0.10
TEST_FRAC      = 0.10
MAX_GROUP_SIZE = 6

# Eval settings
SCORE_BATCH    = 32      # articles per forward pass (ResonantBERT / SBERT)
MAX_TEST_GROUPS = None   # e.g. 500 for a quick smoke run; None = full test split

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print(f"Device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))
print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"Data dir:       {BBC_DIR}")
print(f"Output dir:     {OUTPUT_DIR}")

In [ ]:
# ============================================================
# LOAD DATA (same pipeline + split as the training notebook)
# ============================================================
import yaml
try:
    from yaml import CSafeLoader as YamlLoader   # ~10x faster for 160k files
except ImportError:
    from yaml import SafeLoader as YamlLoader

ARTICLES_DIR = os.path.join(BBC_DIR, "articles")
RANKING_JSON = os.path.join(BBC_DIR, "ranking.json")

assert os.path.exists(RANKING_JSON), f"ranking.json not found at {RANKING_JSON}"
assert os.path.getsize(RANKING_JSON) > 1e6, "ranking.json is an LFS pointer — run `git lfs pull` in the repo root"

_yaml_cache = {}
def load_article(doc_id):
    if doc_id in _yaml_cache: return _yaml_cache[doc_id]
    path = os.path.join(ARTICLES_DIR, f"{doc_id}.yaml")
    if not os.path.exists(path):
        _yaml_cache[doc_id] = None
        return None
    with open(path, encoding="utf-8") as f:
        d = yaml.load(f, Loader=YamlLoader) or {}
    rec = {"headline": str(d.get("title","") or ""), "body": str(d.get("content","") or "")}
    _yaml_cache[doc_id] = rec
    return rec

def build_groups(path, max_size, limit=None):
    groups = []
    with open(path, "r", encoding="utf-8") as f:
        queries = json.load(f)
    for q in queries:
        docs = sorted(q.get("ranking",[]), key=lambda d: int(d["rank"]))[:max_size]
        items = []
        for d in docs:
            art = load_article(d["doc_id"])
            if art is None or not art["body"].strip(): continue
            items.append({"doc_id": d["doc_id"], "headline": art["headline"], "body": art["body"]})
        if len(items) < 2: continue
        for i, it in enumerate(items): it["rank"] = i
        groups.append(items)
        if limit and len(groups) >= limit: break
    return groups

# Parsing ~160k YAML files is slow, so cache the built groups once
GROUPS_CACHE = os.path.join(CACHE_DIR, f"groups_max{MAX_GROUP_SIZE}.pkl")
if os.path.exists(GROUPS_CACHE):
    with open(GROUPS_CACHE, "rb") as f:
        all_groups = pickle.load(f)
    print(f"Loaded groups from cache: {GROUPS_CACHE}")
else:
    print("Building groups from YAML (first run only, takes a few minutes)...")
    all_groups = build_groups(RANKING_JSON, MAX_GROUP_SIZE)
    with open(GROUPS_CACHE, "wb") as f:
        pickle.dump(all_groups, f)

def split_indices(n, seed, val_frac, test_frac):
    idx = list(range(n))
    random.Random(seed).shuffle(idx)
    n_test = max(1, int(round(test_frac * n)))
    n_val  = max(1, int(round(val_frac * n)))
    return sorted(idx[n_test+n_val:]), sorted(idx[n_test:n_test+n_val]), sorted(idx[:n_test])

train_idx, val_idx, test_idx = split_indices(len(all_groups), SEED, VAL_FRAC, TEST_FRAC)
if MAX_TEST_GROUPS:
    test_idx = test_idx[:MAX_TEST_GROUPS]
train_groups = [all_groups[i] for i in train_idx]
test_groups  = [all_groups[i] for i in test_idx]

print(f"Total groups: {len(all_groups)} | Train: {len(train_groups)} | Val: {len(val_idx)} | Test: {len(test_groups)}")
print(f"Test articles: {sum(len(g) for g in test_groups)}")

In [ ]:
# ============================================================
# METRICS (shared across all methods)
# ============================================================
def dcg_at_k(gains, k):
    gains = gains[:k]
    return float((gains / np.log2(np.arange(2, gains.size + 2))).sum())

def ndcg_at_k(pred_scores, gold_ranks, k=5):
    n = len(gold_ranks)
    rel = (n - 1) - gold_ranks
    dcg  = dcg_at_k(rel[np.argsort(-pred_scores)], k)
    idcg = dcg_at_k(rel[np.argsort(-rel)], k)
    return dcg / idcg if idcg > 0 else 0.0

def kendall_tau_metric(pred_scores, gold_ranks):
    if len(pred_scores) < 2: return 0.0
    tau, _ = kendalltau(pred_scores, -gold_ranks)
    return float(tau) if not np.isnan(tau) else 0.0

def pairwise_accuracy(pred_scores, gold_ranks):
    correct, total = 0, 0
    for i in range(len(pred_scores)):
        for j in range(i+1, len(pred_scores)):
            if gold_ranks[i] == gold_ranks[j]: continue
            total += 1
            pred_order = pred_scores[i] > pred_scores[j]
            true_order = gold_ranks[i] < gold_ranks[j]  # lower rank = better
            if pred_order == true_order: correct += 1
    return correct / max(total, 1)

def score_all(score_fn, groups, batch_size):
    """Score every article once, in batches, then split back per group."""
    heads  = [a["headline"] for g in groups for a in g]
    bodies = [a["body"]     for g in groups for a in g]
    out = []
    for i in range(0, len(heads), batch_size):
        out.append(np.asarray(score_fn(heads[i:i+batch_size], bodies[i:i+batch_size]), dtype=np.float64))
        if (i // batch_size) % 200 == 0:
            print(f"    scored {min(i+batch_size, len(heads))}/{len(heads)}", end="\r")
    flat = np.concatenate(out) if out else np.array([])
    per_group, off = [], 0
    for g in groups:
        per_group.append(flat[off:off+len(g)]); off += len(g)
    return per_group

def evaluate_on_groups(score_fn, groups, method_name, batch_size=256, group_ids=None):
    """Unified evaluation. score_fn takes (headlines, bodies) -> scores array.
    Metrics are keyed by group id so significance tests pair the same groups."""
    group_ids = list(range(len(groups))) if group_ids is None else group_ids
    all_scores = score_all(score_fn, groups, batch_size)
    per_group = {}
    for gi, group, scores in zip(group_ids, groups, all_scores):
        ranks = np.array([a["rank"] for a in group])
        if len(scores) < 2: continue
        per_group[gi] = {"ndcg5":  ndcg_at_k(scores, ranks, k=5),
                         "ndcg10": ndcg_at_k(scores, ranks, k=10),
                         "tau":    kendall_tau_metric(scores, ranks),
                         "pa":     pairwise_accuracy(scores, ranks)}
    col = lambda k: np.array([m[k] for m in per_group.values()])
    results = {
        "method": method_name,
        "ndcg@5": col("ndcg5").mean(),   "ndcg@5_std": col("ndcg5").std(),
        "ndcg@10": col("ndcg10").mean(), "ndcg@10_std": col("ndcg10").std(),
        "kendall_tau": col("tau").mean(), "kendall_tau_std": col("tau").std(),
        "pairwise_acc": col("pa").mean(),
        "n_groups": len(per_group),
        "per_group": per_group,
    }
    print(f"  {method_name}: NDCG@5={results['ndcg@5']:.4f}  NDCG@10={results['ndcg@10']:.4f}  "
          f"Tau={results['kendall_tau']:.4f}  PA={results['pairwise_acc']:.4f}  ({results['n_groups']} groups)")
    return results

# Random-order floor, for reference in the tables
_rng = np.random.default_rng(SEED)
def random_score(headlines, bodies):
    return _rng.random(len(headlines))

print("Metrics defined: NDCG@5, NDCG@10 (= full-list NDCG, groups have <=6 items), Kendall Tau, Pairwise Accuracy")
print("Evaluating: Random...")
results_random = evaluate_on_groups(random_score, test_groups, "Random")

In [ ]:
# ============================================================
# BASELINE 1: RULE-BASED (zero learning, keyword heuristics)
# ============================================================

URGENCY_WORDS = ['breaking', 'urgent', 'immediately', 'now', 'just in',
                 'emergency', 'crisis', 'disaster', 'alert', 'exclusive',
                 'developing', 'live', 'update']

EMOTION_WORDS = ['shocking', 'stunning', 'incredible', 'unbelievable', 'outrageous',
                 'furious', 'devastated', 'thrilled', 'amazed', 'horrifying',
                 'heartbreaking', 'terrifying', 'disgusting', 'inspiring']

NOVELTY_WORDS = ['first', 'unprecedented', 'never before', 'new', 'revealed',
                 'exclusive', 'breakthrough', 'discovery', 'groundbreaking',
                 'revolutionary', 'surprising', 'unexpected']

SOCIAL_WORDS  = ['death', 'killed', 'dies', 'war', 'attack', 'election',
                 'president', 'government', 'scandal', 'protest', 'arrested',
                 'charged', 'million', 'billion']

def rule_based_score(headlines, bodies):
    scores = []
    for h, b in zip(headlines, bodies):
        text = (h + " " + b).lower()

        urgency  = sum(1 for w in URGENCY_WORDS if w in text) * 0.25
        emotion  = sum(1 for w in EMOTION_WORDS if w in text) * 0.25
        novelty  = sum(1 for w in NOVELTY_WORDS if w in text) * 0.20
        social   = sum(1 for w in SOCIAL_WORDS  if w in text) * 0.15

        # headline length sweet spot (8-16 words)
        h_len = len(h.split())
        h_score = (1.0 if 8 <= h_len <= 16 else 0.5) * 0.10

        # body length (longer = more substance)
        b_score = min(len(b.split()) / 500.0, 1.0) * 0.05

        total = urgency + emotion + novelty + social + h_score + b_score
        scores.append(total)
    return np.array(scores)

print("Evaluating: Rule-Based...")
results_rule = evaluate_on_groups(rule_based_score, test_groups, "Rule-Based")

In [ ]:
# ============================================================
# BASELINE 2: TF-IDF + RIDGE (classical ML, trained on train split only)
# ============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge

train_texts, train_ranks = [], []
for g in train_groups:
    for a in g:
        train_texts.append(a["headline"] + " " + a["body"][:500])
        train_ranks.append(a["rank"])

print(f"Fitting TF-IDF on {len(train_texts)} train articles...")
tfidf = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1,2))
X_train = tfidf.fit_transform(train_texts)
y_train = np.array(train_ranks, dtype=float)

# Ridge regression on rank (negated so higher score = better)
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, -y_train)

def tfidf_score(headlines, bodies):
    texts = [h + " " + b[:500] for h, b in zip(headlines, bodies)]
    return ridge.predict(tfidf.transform(texts))

print("Evaluating: TF-IDF + Ridge...")
results_tfidf = evaluate_on_groups(tfidf_score, test_groups, "TF-IDF + Ridge", batch_size=4096)

In [ ]:
# ============================================================
# BASELINE 3: SENTENCE-BERT (pretrained, no task-specific training)
# ============================================================
from sentence_transformers import SentenceTransformer

sbert_model = SentenceTransformer('all-MiniLM-L6-v2', device=str(DEVICE))

# Strategy: embed articles, cosine similarity to a "viral content" template centroid
VIRAL_TEMPLATES = [
    "This is shocking breaking news that everyone needs to know about immediately.",
    "An incredible new discovery that changes everything we thought we knew.",
    "Devastating tragedy strikes, leaving millions in shock and mourning.",
    "Major scandal exposed: powerful figures caught in unprecedented controversy.",
    "Revolutionary breakthrough promises to transform daily life forever.",
]
template_centroid = sbert_model.encode(VIRAL_TEMPLATES, convert_to_numpy=True,
                                       normalize_embeddings=True).mean(axis=0)
template_centroid /= np.linalg.norm(template_centroid) + 1e-8

def sbert_score(headlines, bodies):
    texts = [h + " " + b[:300] for h, b in zip(headlines, bodies)]
    emb = sbert_model.encode(texts, convert_to_numpy=True, normalize_embeddings=True,
                             batch_size=128, show_progress_bar=False)
    return emb @ template_centroid

print("Evaluating: Sentence-BERT...")
results_sbert = evaluate_on_groups(sbert_score, test_groups, "Sentence-BERT", batch_size=1024)

In [ ]:
# ============================================================
# BASELINE 4: LLM ZERO-SHOT (prompt-based scoring)
# ============================================================
# Real LLM: one forward pass per article; score = expected value over the
# next-token probabilities of the digits 0-9 (no sampling, deterministic).
# An LLM over the full test set is slow, so it runs on the first LLM_MAX_GROUPS
# test groups; significance tests pair it with the other methods on those groups.
# With USE_LLM = False a keyword heuristic runs instead, and is labelled as such
# so it is never reported as an LLM result.

USE_LLM        = True
LLM_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
LLM_MAX_GROUPS = 500          # None = full test set
LLM_BATCH      = 16

llm_groups   = test_groups[:LLM_MAX_GROUPS] if LLM_MAX_GROUPS else test_groups
llm_group_ids = list(range(len(llm_groups)))

LLM_PROMPT = """Rate the virality potential of this news article on a scale from 0 to 9.
Consider novelty, emotional impact, urgency and shareability.

Headline: {h}
Body: {b}

Answer with a single digit from 0 to 9."""

if USE_LLM:
    from transformers import AutoTokenizer as _AT, AutoModelForCausalLM
    llm_tok = _AT.from_pretrained(LLM_MODEL_NAME)
    llm_tok.padding_side = "left"
    llm = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_NAME, torch_dtype=torch.bfloat16 if DEVICE.type == "cuda" else torch.float32
    ).to(DEVICE).eval()
    digit_ids = [llm_tok.encode(str(d), add_special_tokens=False)[0] for d in range(10)]
    digit_vals = torch.arange(10, dtype=torch.float32, device=DEVICE)

    @torch.no_grad()
    def llm_score(headlines, bodies):
        prompts = [llm_tok.apply_chat_template(
                       [{"role": "user", "content": LLM_PROMPT.format(h=h, b=b[:600])}],
                       tokenize=False, add_generation_prompt=True)
                   for h, b in zip(headlines, bodies)]
        enc = llm_tok(prompts, return_tensors="pt", padding=True).to(DEVICE)
        logits = llm(**enc).logits[:, -1, :].float()
        probs = torch.softmax(logits[:, digit_ids], dim=-1)
        return (probs * digit_vals).sum(-1).cpu().numpy()

    LLM_METHOD = f"LLM Zero-Shot ({LLM_MODEL_NAME.split('/')[-1]})"
else:
    def llm_score(headlines, bodies):
        """Keyword heuristic that mimics an LLM's factor checklist. NOT an LLM."""
        scores = []
        for h, b in zip(headlines, bodies):
            text = (h + " " + b[:400]).lower()
            novel_kw = ['first', 'new', 'discover', 'reveal', 'unprecedented', 'breakthrough']
            novelty = min(sum(0.2 for w in novel_kw if w in text), 1.0)
            emo_kw = ['shock', 'amaz', 'horr', 'delight', 'fury', 'devas', 'inspir',
                      'outrag', 'terr', 'heart', 'tragic', 'triumph']
            emotion = min(sum(0.15 for w in emo_kw if w in text), 1.0)
            urg_kw = ['breaking', 'urgent', 'just', 'now', 'immedia', 'develop', 'live']
            urgency = min(sum(0.2 for w in urg_kw if w in text), 1.0)
            share_kw = ['death', 'kill', 'scandal', 'celebrity', 'health', 'money',
                        'tax', 'child', 'school', 'election', 'president', 'war']
            share = min(sum(0.12 for w in share_kw if w in text), 1.0)
            punct_boost = 0.1 * ('?' in h) + 0.05 * ('!' in h)
            named_ents = min(sum(1 for w in h.split() if w[0:1].isupper() and len(w) > 2) * 0.05, 0.3)
            score = 0.30 * novelty + 0.25 * emotion + 0.20 * urgency + 0.15 * share + punct_boost + named_ents
            scores.append(max(0, min(1, score)))
        return np.array(scores)
    LLM_METHOD = "Keyword CoT Heuristic"

print(f"Evaluating: {LLM_METHOD} on {len(llm_groups)} groups...")
results_llm = evaluate_on_groups(llm_score, llm_groups, LLM_METHOD,
                                 batch_size=LLM_BATCH if USE_LLM else 1024, group_ids=llm_group_ids)

if USE_LLM:  # free VRAM for ResonantBERT
    del llm; torch.cuda.empty_cache() if DEVICE.type == "cuda" else None

In [ ]:
# ============================================================
# OURS: LOAD ResonantBERT v2 + RankingTower v2
# ============================================================
# Class definitions must match the training notebook exactly so that the
# checkpoint keys line up (scoring.*, risk_head.net.*, factor_interaction.encoder.*,
# and the LoRA-wrapped backbone).
from transformers import AutoTokenizer, AutoModelForMaskedLM

class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim=768):
        super().__init__()
        self.attention = nn.Linear(hidden_dim, 1)
    def forward(self, token_reps, attention_mask):
        w = self.attention(token_reps).squeeze(-1)
        w = w.masked_fill(attention_mask == 0, -1e9)
        w = torch.softmax(w, dim=-1)
        return torch.bmm(w.unsqueeze(1), token_reps).squeeze(1)

class SegmentAwareEncoder(nn.Module):
    def __init__(self, backbone_name="answerdotai/ModernBERT-base", hidden_dim=768):
        super().__init__()
        self.backbone = AutoModelForMaskedLM.from_pretrained(backbone_name)
        self.hidden_dim = hidden_dim
        self.pool_headline = AttentionPooling(hidden_dim)
        self.pool_body     = AttentionPooling(hidden_dim)
        self.pool_joint    = AttentionPooling(hidden_dim)
    def forward(self, input_ids, attention_mask, sep_positions):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        tokens = out.hidden_states[-1]
        B, S, D = tokens.shape
        pos = torch.arange(S, device=tokens.device).unsqueeze(0)
        sep = sep_positions.unsqueeze(1)
        h_mask = (pos >= 1) & (pos < sep) & (attention_mask == 1)
        b_mask = (pos > sep) & (attention_mask == 1)
        h_safe = torch.where(h_mask.any(1, keepdim=True), h_mask, attention_mask.bool())
        b_safe = torch.where(b_mask.any(1, keepdim=True), b_mask, attention_mask.bool())
        return {'e_h': self.pool_headline(tokens, h_safe.long()),
                'e_b': self.pool_body(tokens, b_safe.long()),
                'e_j': self.pool_joint(tokens, attention_mask),
                'tokens': tokens, 'attention_mask': attention_mask}

class TokenLevelFactorHead(nn.Module):
    def __init__(self, hidden_dim=768, out_dim=128):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, 1, hidden_dim) * 0.02)
        self.attn = nn.MultiheadAttention(hidden_dim, num_heads=8, batch_first=True)
        self.proj = nn.Sequential(nn.Linear(hidden_dim, out_dim), nn.GELU(), nn.LayerNorm(out_dim))
    def forward(self, tokens, mask):
        q = self.query.expand(tokens.size(0), -1, -1)
        out, _ = self.attn(q, tokens, tokens, key_padding_mask=(mask == 0))
        return self.proj(out.squeeze(1))

class SentenceLevelFactorHead(nn.Module):
    def __init__(self, in_dim=1536, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, 256), nn.GELU(), nn.Dropout(0.1),
                                 nn.Linear(256, out_dim), nn.LayerNorm(out_dim))
    def forward(self, e_h, e_b):
        return self.net(torch.cat([e_h, e_b], dim=-1))

class SurfaceFactorHead(nn.Module):
    def __init__(self, in_dim=768, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, out_dim), nn.GELU())
    def forward(self, e_j):
        return self.net(e_j)

class FactorInteraction(nn.Module):
    def __init__(self, factor_dim=128, num_heads=4, p_drop=0.1):
        super().__init__()
        layer = nn.TransformerEncoderLayer(d_model=factor_dim, nhead=num_heads, dim_feedforward=factor_dim * 2,
                                           dropout=p_drop, batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, num_layers=1)
    def forward(self, F):
        return self.encoder(F)

class DirectScoringHead(nn.Module):
    def __init__(self, num_factors=9, factor_dim=128, embed_dim=128):
        super().__init__()
        self.factor_weights = nn.Parameter(torch.ones(num_factors) / num_factors)
        self.embed_proj = nn.Sequential(nn.Linear(factor_dim, embed_dim), nn.LayerNorm(embed_dim))
        self.score_head = nn.Linear(embed_dim, 1)
    def forward(self, F_prime):
        w = torch.softmax(self.factor_weights, dim=0)
        agg = (F_prime * w.view(1, -1, 1)).sum(dim=1)
        z = F.normalize(self.embed_proj(agg), p=2, dim=-1)
        return {'viral_embedding': z, 'concept_score': self.score_head(z).squeeze(-1), 'factor_weights': w}

class RiskHead(nn.Module):
    def __init__(self, hidden_dim=768):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(hidden_dim, 128), nn.GELU(), nn.Dropout(0.1),
                                 nn.Linear(128, 1), nn.Sigmoid())
    def forward(self, e_j):
        return self.net(e_j).squeeze(-1)

class ResonantBERTv2(nn.Module):
    FACTOR_NAMES = ["novelty","arousal","urgency","social_impact","readability",
                    "specificity","narrative","proportion","affect"]
    FACTOR_TYPES = {"arousal":"token","affect":"token","narrative":"token",
                    "novelty":"sentence","urgency":"sentence","social_impact":"sentence",
                    "specificity":"sentence","proportion":"sentence","readability":"surface"}
    def __init__(self, backbone_name="answerdotai/ModernBERT-base", hidden_dim=768, factor_dim=128):
        super().__init__()
        self.encoder = SegmentAwareEncoder(backbone_name, hidden_dim)
        self.factor_heads = nn.ModuleDict()
        for name in self.FACTOR_NAMES:
            ft = self.FACTOR_TYPES[name]
            if ft == "token": self.factor_heads[name] = TokenLevelFactorHead(hidden_dim, factor_dim)
            elif ft == "sentence": self.factor_heads[name] = SentenceLevelFactorHead(hidden_dim*2, factor_dim)
            else: self.factor_heads[name] = SurfaceFactorHead(hidden_dim, factor_dim)
        self.factor_interaction = FactorInteraction(factor_dim)
        self.scoring = DirectScoringHead(9, factor_dim, factor_dim)
        self.risk_head = RiskHead(hidden_dim)
    def forward(self, input_ids, attention_mask, sep_positions):
        enc = self.encoder(input_ids, attention_mask, sep_positions)
        factors = []
        for name in self.FACTOR_NAMES:
            ft = self.FACTOR_TYPES[name]
            h = self.factor_heads[name]
            if ft == "token": factors.append(h(enc['tokens'], enc['attention_mask']))
            elif ft == "sentence": factors.append(h(enc['e_h'], enc['e_b']))
            else: factors.append(h(enc['e_j']))
        F_prime = self.factor_interaction(torch.stack(factors, dim=1))
        s = self.scoring(F_prime)
        return {'viral_embedding': s['viral_embedding'], 'concept_score': s['concept_score'],
                'factor_weights': s['factor_weights'], 'F_prime': F_prime,
                'risk_score': self.risk_head(enc['e_j'])}

class RankingTowerV2(nn.Module):
    def __init__(self, in_dim=128, p_drop=0.1):
        super().__init__()
        self.tower = nn.Sequential(nn.Linear(in_dim, 64), nn.GELU(), nn.Dropout(p_drop), nn.Linear(64, 1))
        self.temperature = nn.Parameter(torch.tensor(1.0))
    def forward(self, z):
        raw = self.tower(z).squeeze(-1)
        cal = torch.sigmoid(raw / self.temperature)
        return {'raw_score': raw, 'calibrated_score': cal, 'final_score': cal}

def latest_ckpt(pattern):
    """Highest step number, not lexicographic order (BERTv2-1000 > BERTv2-500)."""
    paths = glob.glob(os.path.join(CHECKPOINT_DIR, pattern))
    assert paths, f"No {pattern} found in {CHECKPOINT_DIR}"
    return max(paths, key=lambda p: int(re.search(r"-(\d+)\.pt$", p).group(1)))

# ---- Build model + wrap backbone in LoRA (same config as training) ----
import peft.import_utils
try:
    import peft.tuners.lora.torchao as _lora_torchao
    peft.import_utils.is_torchao_available = lambda: False
    _lora_torchao.is_torchao_available = lambda: False
except Exception: pass
from peft import LoraConfig, get_peft_model

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

print("Loading ResonantBERT v2...")
encoder = ResonantBERTv2(backbone_name=TOKENIZER_NAME)
preferred = ["Wqkv", "Wo", "Wi", "query", "key", "value", "dense"]
present = {n.split(".")[-1] for n, m in encoder.encoder.backbone.named_modules()
           if isinstance(m, nn.Linear) and "decoder" not in n}
targets = [p for p in preferred if p in present] or sorted(present)
encoder.encoder.backbone = get_peft_model(
    encoder.encoder.backbone,
    LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none", target_modules=targets))

bert_path = latest_ckpt("BERTv2-*.pt")
print(f"  Loading: {bert_path}  (LoRA targets: {targets})")
ckpt = torch.load(bert_path, map_location="cpu", weights_only=False)
state = ckpt["model_state"] if "model_state" in ckpt else ckpt
missing, unexpected = encoder.load_state_dict(state, strict=False)
# The checkpoint holds LoRA adapters + all heads; frozen ModernBERT weights come from the hub.
bad_missing = [k for k in missing if "lora_" in k or not k.startswith("encoder.backbone")]
assert not unexpected, f"Unexpected keys (architecture mismatch): {unexpected[:5]}"
assert not bad_missing, f"Trained weights missing from checkpoint: {bad_missing[:5]}"
print(f"  Loaded {len(state)} trained tensors; {len(missing)} frozen backbone tensors from {TOKENIZER_NAME}")
encoder = encoder.to(DEVICE).eval()

print("Loading RankingTower v2...")
ranker = RankingTowerV2(in_dim=EMBEDDING_DIM)
rpath = latest_ckpt("RANKERv2-*.pt")
print(f"  Loading: {rpath}")
rckpt = torch.load(rpath, map_location="cpu", weights_only=False)
ranker.load_state_dict(rckpt["model_state"] if "model_state" in rckpt else rckpt, strict=True)
ranker = ranker.to(DEVICE).eval()

print("\nResonantBERT v2 + RankingTower v2 loaded")

In [ ]:
# ============================================================
# EVALUATE ResonantBERT v2 + RankingTower v2
# ============================================================
_amp = DEVICE.type == "cuda"
sep_token = tokenizer.sep_token or "[SEP]"
sep_id = tokenizer.sep_token_id or tokenizer.convert_tokens_to_ids(sep_token)

@torch.no_grad()
def encode_batch(headlines, bodies):
    texts = [f"{h} {sep_token} {b}" for h, b in zip(headlines, bodies)]
    enc = tokenizer(texts, padding=True, truncation=True, max_length=MAX_SEQ_LEN, return_tensors="pt")
    sep_positions = []
    for ids in enc["input_ids"]:
        locs = (ids == sep_id).nonzero(as_tuple=True)[0]
        sep_positions.append(locs[0].item() if len(locs) >= 1 else 1)
    with torch.autocast("cuda", enabled=_amp):
        return encoder(enc["input_ids"].to(DEVICE), enc["attention_mask"].to(DEVICE),
                       torch.tensor(sep_positions, dtype=torch.long, device=DEVICE))

def resonant_score(headlines, bodies):
    z = encode_batch(headlines, bodies)["viral_embedding"].float()
    return ranker(z)["final_score"].cpu().numpy()

def resonant_concept_score(headlines, bodies):
    """Ablation: encoder's own concept_score head, no RankingTower."""
    return encode_batch(headlines, bodies)["concept_score"].float().cpu().numpy()

print("Evaluating: ResonantBERT v2 + RankingTower v2...")
results_sota = evaluate_on_groups(resonant_score, test_groups, "ResonantBERT v2", batch_size=SCORE_BATCH)
print("Evaluating: ResonantBERT v2 (concept_score, no tower)...")
results_sota_concept = evaluate_on_groups(resonant_concept_score, test_groups,
                                          "ResonantBERT v2 (no tower)", batch_size=SCORE_BATCH)

In [ ]:
# ============================================================
# RESULTS TABLE + STATISTICAL SIGNIFICANCE
# ============================================================
all_results = [r for r in [results_random, results_rule, results_tfidf, results_sbert, results_llm,
                           globals().get("results_sota_concept"), globals().get("results_sota")] if r]

print("\n" + "=" * 96)
print("TABLE 1: MAIN RESULTS (Test Set)")
print("=" * 96)
print(f"{'Method':<32} {'NDCG@5':>14} {'NDCG@10':>14} {'Kendall τ':>14} {'Pairwise':>9} {'Groups':>7}")
print("-" * 96)
for r in sorted(all_results, key=lambda x: -x['ndcg@10']):
    print(f"{r['method']:<32} "
          f"{r['ndcg@5']:.4f}±{r['ndcg@5_std']:.3f}  "
          f"{r['ndcg@10']:.4f}±{r['ndcg@10_std']:.3f}  "
          f"{r['kendall_tau']:+.4f}±{r['kendall_tau_std']:.3f}  "
          f"{r['pairwise_acc']:.4f}  "
          f"{r['n_groups']:>6}")
print("=" * 96)

# Significance: paired tests on per-group NDCG@10, over the groups both methods scored
if "results_sota" in globals():
    print("\n" + "=" * 90)
    print("TABLE 2: STATISTICAL SIGNIFICANCE vs. ResonantBERT v2 (paired, per-group NDCG@10)")
    print("=" * 90)
    print(f"{'Baseline':<32} {'Δ NDCG@10':>10} {'Cohen d':>9} {'t-test p':>11} {'Wilcoxon p':>11} {'Sig':>4} {'n':>6}")
    print("-" * 90)
    for r in all_results:
        if r is results_sota: continue
        common = sorted(set(results_sota['per_group']) & set(r['per_group']))
        s = np.array([results_sota['per_group'][g]['ndcg10'] for g in common])
        b = np.array([r['per_group'][g]['ndcg10'] for g in common])
        diff = s - b
        _, p_t = ttest_rel(s, b)
        try:    _, p_w = wilcoxon(s, b)
        except ValueError: p_w = float("nan")   # all differences zero
        cohens_d = diff.mean() / (diff.std(ddof=1) + 1e-8)
        sig = "***" if p_t < 0.001 else ("**" if p_t < 0.01 else ("*" if p_t < 0.05 else "ns"))
        print(f"{r['method']:<32} {diff.mean():>+10.4f} {cohens_d:>9.3f} {p_t:>11.2e} {p_w:>11.2e} {sig:>4} {len(common):>6}")
    print("=" * 90)

In [ ]:
# ============================================================
# VISUALIZATIONS
# ============================================================
is_ours = lambda m: 'Resonant' in m
ndcg10_values = lambda r: [m['ndcg10'] for m in r['per_group'].values()]

# Figure 1: NDCG@10 bar chart
fig, ax = plt.subplots(figsize=(10, 5))
ordered = sorted(all_results, key=lambda x: x['ndcg@10'])
methods = [r['method'] for r in ordered]
ndcg10s = [r['ndcg@10'] for r in ordered]
stds    = [r['ndcg@10_std'] for r in ordered]
colors  = ['#2ecc71' if is_ours(m) else '#95a5a6' for m in methods]
bars = ax.barh(methods, ndcg10s, xerr=stds, color=colors, alpha=0.8, edgecolor='black', capsize=3)
ax.set_xlabel('NDCG@10', fontsize=12)
ax.set_title('Virality Ranking: Model Comparison', fontsize=14, fontweight='bold')
for bar, score in zip(bars, ndcg10s):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{score:.4f}', va='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig1_ndcg_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

# Figure 2: Per-group NDCG@10 distribution (box plot)
fig, ax = plt.subplots(figsize=(10, 5))
ordered = sorted(all_results, key=lambda x: -x['ndcg@10'])
labels = [r['method'] for r in ordered]
bp = ax.boxplot([ndcg10_values(r) for r in ordered], tick_labels=labels, patch_artist=True, vert=True)
for patch, label in zip(bp['boxes'], labels):
    patch.set_facecolor('#2ecc71' if is_ours(label) else '#bdc3c7')
    patch.set_alpha(0.7)
ax.set_ylabel('NDCG@10', fontsize=12)
ax.set_title('Per-Group Performance Variance', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig2_variance.png'), dpi=300, bbox_inches='tight')
plt.show()

# Figure 3: all metrics
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, metric, title in zip(axes, ['ndcg@5', 'ndcg@10', 'kendall_tau'], ['NDCG@5', 'NDCG@10', 'Kendall τ']):
    ordered = sorted(all_results, key=lambda x: x[metric])
    vals  = [r[metric] for r in ordered]
    names = [r['method'] for r in ordered]
    ax.barh(names, vals, color=['#2ecc71' if is_ours(n) else '#95a5a6' for n in names], alpha=0.8)
    ax.set_title(title, fontsize=12, fontweight='bold')
    lo, hi = min(0, min(vals)), max(0, max(vals))   # Kendall τ can be negative
    ax.set_xlim([lo * 1.15, hi * 1.15 if hi > 0 else 0.05])
plt.suptitle('All Metrics Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig3_all_metrics.png'), dpi=300, bbox_inches='tight')
plt.show()

print("All figures saved to:", OUTPUT_DIR)

In [ ]:
# ============================================================
# EXPORT: LaTeX Tables + JSON
# ============================================================
latex = "\\begin{table}[t]\n\\centering\n\\small\n"
latex += "\\begin{tabular}{lcccc}\n\\toprule\n"
latex += "\\textbf{Model} & \\textbf{NDCG@5} & \\textbf{NDCG@10} & \\textbf{Kendall $\\tau$} & \\textbf{Pairwise Acc.} \\\\\n"
latex += "\\midrule\n"
for r in sorted(all_results, key=lambda x: -x['ndcg@10']):
    name = r['method'].replace("_", "\\_")
    name = f"\\textbf{{{name}}}" if is_ours(r['method']) else name
    latex += (f"{name} & {r['ndcg@5']:.3f} $\\pm$ {r['ndcg@5_std']:.3f} & "
              f"{r['ndcg@10']:.3f} $\\pm$ {r['ndcg@10_std']:.3f} & "
              f"{r['kendall_tau']:.3f} $\\pm$ {r['kendall_tau_std']:.3f} & "
              f"{r['pairwise_acc']:.3f} \\\\\n")
latex += "\\bottomrule\n\\end{tabular}\n"
latex += "\\caption{Virality ranking results on the held-out BBC test split.}\n"
latex += "\\label{tab:main}\n\\end{table}"

with open(os.path.join(OUTPUT_DIR, "table1_main.tex"), "w", encoding="utf-8") as f:
    f.write(latex)
print("Saved: table1_main.tex")

results_export = {r['method']: {k: (float(v) if isinstance(v, (np.floating, float)) else v)
                                for k, v in r.items() if k != 'per_group'}
                  for r in all_results}
with open(os.path.join(OUTPUT_DIR, "results.json"), "w", encoding="utf-8") as f:
    json.dump(results_export, f, indent=2, default=str)
print("Saved: results.json")

print("\n" + "=" * 60)
print("EVALUATION COMPLETE")
print("=" * 60)
print(f"Output directory: {OUTPUT_DIR}")
print(f"Files: {sorted(os.listdir(OUTPUT_DIR))}")
if "results_sota" in globals():
    best_baseline = max([r for r in all_results if not is_ours(r['method'])], key=lambda x: x['ndcg@10'])
    improvement = (results_sota['ndcg@10'] - best_baseline['ndcg@10']) / best_baseline['ndcg@10'] * 100
    print(f"\nResonantBERT v2: {results_sota['ndcg@10']:.4f} NDCG@10")
    print(f"vs. best baseline ({best_baseline['method']}): {improvement:+.1f}%")